# Chapter 18: 멀티 에이전트 아키텍처 (Multi-Agent Architectures)

여러 AI 에이전트가 협력하는 멀티 에이전트 시스템을 설계합니다.

**Sections:**
- 18.0 Setup & Environment
- 18.1 Network Architecture (P2P 에이전트 간 직접 전환)
- 18.2 Supervisor Architecture (중앙 조정자)
- 18.3 Supervisor as Tools (에이전트를 도구로 캡슐화)

---
## 18.0 Setup & Environment

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

print("OPENAI_API_KEY:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL"))
print("OPENAI_MODEL_NAME:", os.getenv("OPENAI_MODEL_NAME"))

OPENAI_API_KEY: a703ad29...
OPENAI_BASE_URL: https://lgcns-assetization3-japan-east.openai.azure.com/openai/v1
OPENAI_MODEL_NAME: gpt-5.1


In [2]:
from importlib.metadata import version

print(f"langgraph: {version('langgraph')}")
print(f"langchain: {version('langchain')}")

langgraph: 1.1.3
langchain: 1.2.13


---
## 18.1 Network Architecture — P2P 에이전트 전환

중앙 조정자 없이 각 에이전트가 **자율적으로** 다른 에이전트에게 대화를 넘김:

```
korean_agent ◄──► greek_agent
      ▲               ▲
      └──► spanish_agent ◄──┘
```

핵심:
- `handoff_tool` — `Command(goto=..., graph=Command.PARENT)` 로 에이전트 전환
- `make_agent()` — 에이전트 팩토리 함수 (서브그래프 생성)
- 각 에이전트는 독립적인 ReAct 루프를 가짐

In [3]:
import os
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


# 1단계: 상태 정의
class AgentsState(MessagesState):
    current_agent: str
    transfered_by: str


# 2단계: 에이전트 팩토리 — 동일한 구조의 에이전트를 매개변수만 바꿔서 생성
def make_agent(prompt, tools):
    def agent_node(state: AgentsState):
        llm_with_tools = llm.bind_tools(tools)
        system_msg = f"{prompt}\n\nIf the customer writes in a language you cannot assist with, use the handoff_tool to transfer to the correct agent. Do not transfer to yourself."
        messages = [{"role": "system", "content": system_msg}] + state["messages"]
        response = llm_with_tools.invoke(messages)
        return {"messages": [response]}

    agent_builder = StateGraph(AgentsState)
    agent_builder.add_node("agent", agent_node)
    agent_builder.add_node("tools", ToolNode(tools=tools))

    agent_builder.add_edge(START, "agent")
    agent_builder.add_conditional_edges("agent", tools_condition)
    agent_builder.add_edge("tools", "agent")

    return agent_builder.compile()


# 3단계: 핸드오프 도구 — Command.PARENT로 부모 그래프에서 전환
@tool
def handoff_tool(transfer_to: str, transfered_by: str):
    """
    Handoff to another agent.
    Use this when the customer speaks a language you cannot assist with.

    Possible values for transfer_to: korean_agent, greek_agent, spanish_agent
    Possible values for transfered_by: korean_agent, greek_agent, spanish_agent

    Args:
        transfer_to: The agent to transfer the conversation to
        transfered_by: The agent that transferred the conversation
    """
    # 자기 자신에게 전환하는 무한 루프 방어
    if transfer_to == transfered_by:
        return {"error": "Cannot transfer to yourself. Please respond to the customer directly."}

    return Command(
        update={
            "current_agent": transfer_to,
            "transfered_by": transfered_by,
        },
        goto=transfer_to,
        graph=Command.PARENT,  # 부모 그래프에서 전환!
    )


# 4단계: 최상위 그래프 조립
graph_builder = StateGraph(AgentsState)

graph_builder.add_node(
    "korean_agent",
    make_agent(
        prompt="You are a helpful Korean-speaking customer support agent. Please respond to customers in Korean.",
        tools=[handoff_tool],
    ),
    destinations=("greek_agent", "spanish_agent"),
)
graph_builder.add_node(
    "greek_agent",
    make_agent(
        prompt="You are a helpful Greek-speaking customer support agent. Please respond to customers in Greek.",
        tools=[handoff_tool],
    ),
    destinations=("korean_agent", "spanish_agent"),
)
graph_builder.add_node(
    "spanish_agent",
    make_agent(
        prompt="You are a helpful Spanish-speaking customer support agent. Please respond to customers in Spanish.",
        tools=[handoff_tool],
    ),
    destinations=("greek_agent", "korean_agent"),
)

graph_builder.add_edge(START, "korean_agent")

graph = graph_builder.compile()

In [4]:
# 한국어로 시작 — korean_agent가 직접 처리
print("=== 한국어 메시지 ===")
for event in graph.stream(
    {"messages": [{"role": "user", "content": "안녕하세요! 계정 문제가 있어요."}]},
    stream_mode="updates",
):
    for key, value in event.items():
        print(f"  [{key}] current_agent={value.get('current_agent', '-')}")

=== 한국어 메시지 ===
  [korean_agent] current_agent=-


In [5]:
# 스페인어로 시작 — korean_agent가 감지 → spanish_agent로 전환
print("=== 스페인어 메시지 ===")
for event in graph.stream(
    {"messages": [{"role": "user", "content": "Hola! Necesito ayuda con mi cuenta."}]},
    stream_mode="updates",
):
    for key, value in event.items():
        print(f"  [{key}] current_agent={value.get('current_agent', '-')}")

=== 스페인어 메시지 ===
  [korean_agent] current_agent=spanish_agent
  [spanish_agent] current_agent=spanish_agent


### Exercise 18.1

1. 그리스어 메시지를 보내 전환 흐름을 확인하라.
2. 새로운 언어 에이전트(예: 일본어)를 추가해 보라. `handoff_tool`의 docstring도 함께 수정해야 한다.
3. `Command.PARENT`를 제거하면 어떤 오류가 발생하는지 실험해 보라.

In [6]:
# Try it yourself


---
## 18.2 Supervisor Architecture — 중앙 조정자

하나의 **슈퍼바이저**가 모든 라우팅 결정을 담당:

```
         Supervisor
        /    |     \
   korean  greek  spanish
        \    |     /
         Supervisor  ← 다시 돌아옴
```

장점:
- 에이전트는 라우팅 몰라도 됨 (자기 역할에만 집중)
- 확장 시 **슈퍼바이저만 수정**
- `reasoning` 필드로 디버깅 용이

In [7]:
import os
from typing import Literal
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


# 슈퍼바이저 출력 스키마
class SupervisorOutput(BaseModel):
    next_agent: Literal["korean_agent", "spanish_agent", "greek_agent", "__end__"]
    reasoning: str


class AgentsState(MessagesState):
    current_agent: str
    transfered_by: str
    reasoning: str


# 에이전트 팩토리 — 라우팅 로직 없이 단순 응답만
def make_agent(prompt):
    def agent_node(state: AgentsState):
        messages = [{"role": "system", "content": prompt}] + state["messages"]
        response = llm.invoke(messages)
        return {"messages": [response]}

    agent_builder = StateGraph(AgentsState)
    agent_builder.add_node("agent", agent_node)
    agent_builder.add_edge(START, "agent")
    agent_builder.add_edge("agent", END)
    return agent_builder.compile()


# 슈퍼바이저 노드 — Structured Output으로 안전한 라우팅
def supervisor(state: AgentsState):
    structured_llm = llm.with_structured_output(SupervisorOutput)
    response = structured_llm.invoke(
        [
            {
                "role": "system",
                "content": """You are a supervisor that routes conversations to the appropriate language agent.

Analyse the customer's request and decide which agent should handle the conversation.

Options: greek_agent, spanish_agent, korean_agent

Rules:
- Never transfer to the same agent twice in a row.
- If an agent has already replied, end the conversation by returning __end__"""
            },
            {
                "role": "user",
                "content": f"Conversation so far: {state.get('messages', [])}"
            }
        ]
    )
    print(f"  Supervisor → {response.next_agent} (reason: {response.reasoning[:60]})")
    return Command(
        goto=response.next_agent,
        update={"reasoning": response.reasoning},
    )


# 그래프 조립 — 순환 구조
graph_builder = StateGraph(AgentsState)

graph_builder.add_node(
    "supervisor", supervisor,
    destinations=("korean_agent", "spanish_agent", "greek_agent", END),
)
graph_builder.add_node("korean_agent", make_agent(
    prompt="You are a helpful Korean-speaking customer support agent. Please respond in Korean.",
))
graph_builder.add_node("greek_agent", make_agent(
    prompt="You are a helpful Greek-speaking customer support agent. Please respond in Greek.",
))
graph_builder.add_node("spanish_agent", make_agent(
    prompt="You are a helpful Spanish-speaking customer support agent. Please respond in Spanish.",
))

# 순환: START → supervisor → agent → supervisor → ... → END
graph_builder.add_edge(START, "supervisor")
graph_builder.add_edge("korean_agent", "supervisor")
graph_builder.add_edge("spanish_agent", "supervisor")
graph_builder.add_edge("greek_agent", "supervisor")

graph = graph_builder.compile()

In [8]:
# 한국어 메시지
print("=== 한국어 메시지 ===")
for event in graph.stream(
    {"messages": [{"role": "user", "content": "안녕하세요, 비밀번호를 바꾸고 싶어요."}]},
    stream_mode="updates",
):
    for key, value in event.items():
        print(f"  [{key}] reasoning={value.get('reasoning', '-')[:80]}")

=== 한국어 메시지 ===
  Supervisor → korean_agent (reason: User is speaking in Korean and asking for help changing a pa)
  [supervisor] reasoning=User is speaking in Korean and asking for help changing a password, so the korea
  [korean_agent] reasoning=User is speaking in Korean and asking for help changing a password, so the korea
  Supervisor → __end__ (reason: A korean_agent has already replied in the conversation, so a)
  [supervisor] reasoning=A korean_agent has already replied in the conversation, so according to the rule


In [9]:
# 스페인어 메시지 → 슈퍼바이저가 spanish_agent로 라우팅
print("=== 스페인어 메시지 ===")
for event in graph.stream(
    {"messages": [{"role": "user", "content": "Hola! Necesito ayuda con mi cuenta."}]},
    stream_mode="updates",
):
    for key, value in event.items():
        print(f"  [{key}] reasoning={value.get('reasoning', '-')[:80]}")

=== 스페인어 메시지 ===
  Supervisor → spanish_agent (reason: El usuario se comunica en español y pide ayuda con su cuenta)
  [supervisor] reasoning=El usuario se comunica en español y pide ayuda con su cuenta; el agente más adec
  [spanish_agent] reasoning=El usuario se comunica en español y pide ayuda con su cuenta; el agente más adec
  Supervisor → __end__ (reason: A spanish_agent has already replied earlier in the conversat)
  [supervisor] reasoning=A spanish_agent has already replied earlier in the conversation, so according to


### Exercise 18.2

1. `reasoning` 필드를 출력하여 슈퍼바이저가 어떤 근거로 라우팅하는지 분석하라.
2. 에이전트를 추가할 때 네트워크 vs 슈퍼바이저 아키텍처에서 각각 어떤 코드를 수정해야 하는지 비교하라.
3. `SupervisorOutput`에서 `__end__` 옵션을 제거하면 어떤 일이 벌어지는지 실험하라.

In [10]:
# Try it yourself


---
## 18.3 Supervisor as Tools — 에이전트를 도구로 캡슐화

에이전트를 `@tool`로 변환 → 슈퍼바이저가 LLM의 **tool calling**으로 에이전트 호출:

```
Supervisor ──tools_condition──► ToolNode
                               ├ korean_agent
                               ├ greek_agent
                               └ spanish_agent
```

장점:
- 별도 라우팅 로직 불필요 (LLM이 알아서 도구 선택)
- 가장 깔끔한 구조
- 에이전트 추가 = 도구 추가

In [11]:
import os
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


class State(MessagesState):
    pass


# 에이전트를 도구로 캡슐화!
@tool
def korean_agent(message: str) -> str:
    """Transfer to Korean customer support agent. Use when the customer speaks Korean."""
    response = llm.invoke(
        f"You're a Korean customer support agent. Respond in Korean.\nCustomer: {message}"
    )
    return response.content


@tool
def greek_agent(message: str) -> str:
    """Transfer to Greek customer support agent. Use when the customer speaks Greek."""
    response = llm.invoke(
        f"You're a Greek customer support agent. Respond in Greek.\nCustomer: {message}"
    )
    return response.content


@tool
def spanish_agent(message: str) -> str:
    """Transfer to Spanish customer support agent. Use when the customer speaks Spanish."""
    response = llm.invoke(
        f"You're a Spanish customer support agent. Respond in Spanish.\nCustomer: {message}"
    )
    return response.content


# 슈퍼바이저 = LLM + 도구
agent_tools = [korean_agent, greek_agent, spanish_agent]
llm_with_tools = llm.bind_tools(agent_tools)


def supervisor(state: State):
    response = llm_with_tools.invoke(
        [
            {
                "role": "system",
                "content": "You are a customer support supervisor. "
                "Route the customer to the appropriate language agent using the tools.",
            }
        ]
        + state["messages"]
    )
    return {"messages": [response]}


# 그래프 — 깔끔한 ReAct 구조
graph_builder = StateGraph(State)

graph_builder.add_node("supervisor", supervisor)
graph_builder.add_node("tools", ToolNode(tools=agent_tools))

graph_builder.add_edge(START, "supervisor")
graph_builder.add_conditional_edges("supervisor", tools_condition)
graph_builder.add_edge("tools", "supervisor")

graph = graph_builder.compile()

In [12]:
# 한국어 → korean_agent 도구 호출
print("=== 한국어 ===")
result = graph.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요, 비밀번호를 변경하고 싶습니다."}]}
)
print(result["messages"][-1].content[:200])

=== 한국어 ===
비밀번호 변경 도와드리겠습니다.  

비밀번호를 어디에서 변경하고 싶으신지 알려주세요:

1. 어떤 서비스/사이트인가요?  
   - 예: 이메일(네이버, 구글 등), 쇼핑몰, 은행, 회사 계정, 게임 등  
2. 어디에서 사용 중이신가요?  
   - 웹사이트(브라우저), 모바일 앱, PC 프로그램 등  
3. 사용 기기/환경은 무엇인가요?  
   - 


In [13]:
# 스페인어 → spanish_agent 도구 호출
print("=== 스페인어 ===")
result = graph.invoke(
    {"messages": [{"role": "user", "content": "Hola, necesito cambiar mi contraseña."}]}
)
print(result["messages"][-1].content[:200])

=== 스페인어 ===
¡Hola! Te ayudo con gusto a cambiar tu contraseña.

Primero necesito un detalle más:  

1) ¿Es la contraseña de tu cuenta de acceso a nuestra web/app?  
2) ¿O de otro servicio (por ejemplo, correo, pa


### Exercise 18.3

1. 일본어 에이전트를 `@tool`로 추가해 보라. 코드 수정이 얼마나 간단한지 느껴 보라.
2. 3가지 아키텍처(Network, Supervisor, Supervisor+Tools)를 비교 정리하라:
   - 라우팅 방식
   - 에이전트 복잡도
   - 확장성
   - 디버깅 용이성
3. 언어 라우팅 외에 실무 시나리오(예: 부서별 라우팅, 기술 스택별 분류)를 설계해 보라.

In [14]:
# Try it yourself


---
## Architecture Comparison — 아키텍처 비교

| | Network (18.1) | Supervisor (18.2) | Supervisor+Tools (18.3) |
|--|---------|------------|------------------|
| **라우팅** | 각 에이전트가 자율 | 중앙 슈퍼바이저 | LLM tool calling |
| **에이전트 복잡도** | 높음 (handoff 로직) | 낮음 (응답만) | 최소 (@tool 함수) |
| **확장성** | 모든 에이전트 수정 | 슈퍼바이저만 수정 | 도구 추가만 |
| **디버깅** | 어려움 | reasoning 추적 | tool call 추적 |
| **적합한 경우** | 에이전트 소수, 자율성 필요 | 중규모, 제어 필요 | 대규모, 깔끔한 구조 |

---
## Final Exercises — 종합 실습

### 과제 1: 4개 언어 Network 아키텍처 (★★☆)

기존 3개 에이전트에 **일본어 에이전트**를 추가한 Network 아키텍처를 구현하라.
`handoff_tool`의 docstring과 `destinations`도 수정해야 한다.

In [15]:
# 과제 1: 여기에 작성


### 과제 2: 부서별 Supervisor 라우팅 (★★★)

언어 대신 **부서별** 라우팅 슈퍼바이저:
- `billing_agent` — 결제/환불 문의
- `tech_agent` — 기술 지원
- `general_agent` — 일반 문의

슈퍼바이저가 `reasoning`과 함께 적절한 부서로 라우팅.

In [16]:
# 과제 2: 여기에 작성


### 과제 3: Supervisor+Tools + 전문 도구 (★★★)

Supervisor+Tools 아키텍처에 **전문 도구**를 가진 에이전트:
- `weather_agent` — 날씨 조회 도구 보유
- `calculator_agent` — 계산 도구 보유
- `search_agent` — 검색 도구 보유

슈퍼바이저가 질문 내용에 따라 적절한 에이전트 도구를 호출.

In [17]:
# 과제 3: 여기에 작성
